# JS09 - Lab 3
## Exact NN vs. HNSW

Prosedur yang sama seperti lab sebelumnya, namun pembandingnya adalah **HNSW**
(*Hierarchical Navigable Small World*) melalui library `hnswlib`. HNSW membangun graf berlapis
(*multi-layer graph*) dan menelusurinya secara *greedy* saat query.

### Step 1 - Install Library

In [1]:
!pip install hnswlib

### Step 2 - Import Libraries

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

### Step 3 - Create Dummy Dataset

In [3]:
# Build Dummy Dataset
np.random.seed(42)
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # search for 5 nearest neighbors

print("Dataset shape:", data.shape)

Dataset shape: (1000, 2)


### Step 4 - Construct NN Model

`algorithm='brute'` memaksa `NearestNeighbors` menghitung jarak ke seluruh data - hasilnya
menjadi *ground truth* untuk mengukur recall HNSW.

In [4]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performance
start = time.perf_counter()
distances, indices = nn.kneighbors(query)
end = time.perf_counter()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Time:", end - start, "s")

=== Exact NN ===
Indices: [[137 170 750 316 991]]
Distances: [[0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]]
Time: 0.04471980000016629 s


### Step 5 - Construct HNSW Model

Parameter penting:

* `M = 16` - jumlah koneksi per node pada graf.
* `ef_construction = 100` - ukuran kandidat saat membangun graf (menentukan kualitas index).
* `ef = 50` - ukuran kandidat saat query; **trade-off** utama antara kecepatan dan akurasi.

In [5]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.perf_counter()
labels, distances = p.knn_query(query, k=k)
end = time.perf_counter()

print("=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Time:", end - start, "s")

=== HNSW ===
Indices: [[137 170 750 316 991]]
Distances: [[0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]]
Time: 0.0001432999997632578 s


### Step 6 - Compare the Results

HNSW dengan `space='l2'` mengembalikan **jarak kuadrat** (bukan akar kuadrat), sehingga angkanya
lebih kecil daripada jarak Euclidean exact. Urutan tetangganya tetap sama, jadi yang dibandingkan
adalah **indeks** tetangga dan waktu pencarian.

In [6]:
print("Exact indices:", list(indices[0]))
print("HNSW  indices:", list(labels[0]))
print("Recall@5:", len(set(indices[0]) & set(labels[0])) / k)

Exact indices: [np.int64(137), np.int64(170), np.int64(750), np.int64(316), np.int64(991)]
HNSW  indices: [np.uint64(137), np.uint64(170), np.uint64(750), np.uint64(316), np.uint64(991)]
Recall@5: 1.0


### Step 7 - Effect of the `ef` Parameter

`ef` menentukan berapa banyak kandidat yang diperiksa saat menelusuri graf. Nilai kecil berarti
pencarian lebih cepat tetapi recall turun; nilai besar sebaliknya.

In [7]:
rows = []
for ef in [10, 20, 50, 100, 200]:
    p.set_ef(ef)
    start = time.perf_counter()
    for _ in range(500):
        lab, dist = p.knn_query(query, k=k)
    elapsed = (time.perf_counter() - start) / 500 * 1000   # ms/query
    rec = len(set(indices[0]) & set(lab[0])) / k
    rows.append((ef, rec, elapsed))

print(f"{'ef':>5} | {'recall@5':>8} | {'time (ms)':>10}")
print("-" * 32)
for ef, rec, elapsed in rows:
    print(f"{ef:>5} | {rec:>8.2f} | {elapsed:>10.4f}")

   ef | recall@5 |  time (ms)
--------------------------------
   10 |     1.00 |     0.0086
   20 |     1.00 |     0.0100
   50 |     1.00 |     0.0134
  100 |     1.00 |     0.0201
  200 |     1.00 |     0.0305
